## Ноутбук с подготовкой и очисткой данных

1. Импорт библиотек и конфигурация проекта

In [1]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
}

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pyarrow

2. Загрузка и первичный осмотр данных

In [3]:
dates_only = pd.read_csv('../data/raw/raw_dataset.csv', usecols=['Дата размещения объявления'])
print("Самая поздняя дата в файле:", dates_only['Дата размещения объявления'].max())

Самая поздняя дата в файле: 2025-07-14


Заметим, что самая поздняя дата объявления - 2025-07-14. Так как файл слишком большой (5гб), отберём только объявления, размещенные с 2025-01-14 по 2025-07-14: это позволит не только облегчить вычисления, но и сделает будущую модель лучше, ведь она будет обучена на относительно "свежих" данных (учитываем инфляцию и актуальность цен).

In [5]:
date = 'Дата размещения объявления'
chunks = []
for chunk in pd.read_csv('../data/raw/raw_dataset.csv', chunksize = 100000, low_memory=False):
    filtered = chunk[chunk[date].between('2025-01-14','2025-07-14')]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index = True)
df.shape

(585855, 58)

585855 строк - оптимальное значенение для обучения модели. Больше брать смысла нет, так как качество модели растет логарифмически по отношению к объему данных. Для начала проверим, есть ли в нашем датасете информация о спецтехнике.

In [6]:
trucks_count = df['Тип техники'].notna().sum()
print(f"Найдено коммерческой техники/спецтехники: {trucks_count} шт.")

Найдено коммерческой техники/спецтехники: 0 шт.


Отлично! Никаких грузовиков, тягачей и кранов в нашем полном датасете нет - можно спокойно работать с нашим последующим сэмплом в 100к, не боясь что мы удалим важные столбцы для нелегковых автомобилей.
Все эксперименты будем проводить на DEV_MODE = True, чтобы работать с 100тыс. строк. В конце работы в CONFIG поменяем значение на False -> финальный запуск на всем объеме (585к строк)

In [7]:
if CONFIG['DEV_MODE']:
    df = df.sample(n=100000, random_state=CONFIG['RANDOM_STATE'])
    print("Режим разработки (100к строк). Всё будет летать!")
else:
    df = df.copy()
    print("Финальный режим (585к строк). Обучаем итоговую модель.")

Финальный режим (585к строк). Обучаем итоговую модель.


In [8]:
df.info

<bound method DataFrame.info of              Название машины     Год  \
0       Aston Martin Vantage  2018.0   
1           Aston Martin DB9  2005.0   
2          Aston Martin DB11  2017.0   
3           Aston Martin DB9  2013.0   
4           Aston Martin DBS  2019.0   
...                      ...     ...   
585850            Volvo XC90  2006.0   
585851             Volvo C30  2008.0   
585852             Volvo V90  2019.0   
585853             Volvo V60  2018.0   
585854             Volvo S80  2002.0   

                                                   Ссылка  \
0       https://auto.drom.ru/himki/aston_martin/vantag...   
1       https://auto.drom.ru/krasnodar/aston_martin/db...   
2       https://auto.drom.ru/moscow/aston_martin/db11/...   
3       https://auto.drom.ru/moscow/aston_martin/db9/1...   
4       https://auto.drom.ru/moscow/aston_martin/dbs/5...   
...                                                   ...   
585850  https://auto.drom.ru/moscow/volvo/xc90/5051113...   

In [9]:
df.head(5)

,Название машины,Год,Ссылка,Дата размещения объявления,Цена,Кол-во просмотров,Скрыто,Объем двигателя,Тип двигателя,Мощность,...,Объем ковша,Длина стрелы,Грузоподъемность стрелы,Высота вышки,Состояние,Страна производства,Высота подъема,Ошибка_ст,Ошибка_знач,Пропуски в данных
0,Aston Martin Vantage,2018.0,https://auto.drom.ru/himki/aston_martin/vantag...,2025-04-03,11834000.0,1899.0,0.0,4.0,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,","открытый,","17,"
1,Aston Martin DB9,2005.0,https://auto.drom.ru/krasnodar/aston_martin/db...,2025-04-18,4999000.0,1230.0,0.0,5.9,бензин,456.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"9,","510.0,","17,"
2,Aston Martin DB11,2017.0,https://auto.drom.ru/moscow/aston_martin/db11/...,2025-05-16,13900000.0,1317.0,0.0,5.2,бензин,608.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aston Martin DB9,2013.0,https://auto.drom.ru/moscow/aston_martin/db9/1...,2025-03-29,8500000.0,14571.0,0.0,5.9,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aston Martin DBS,2019.0,https://auto.drom.ru/moscow/aston_martin/dbs/5...,2025-03-31,24300000.0,11888.0,0.0,5.2,бензин,715.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
(df.isnull().mean() * 100).round(2)

Название машины                 0.00
Год                             0.00
Ссылка                          0.00
Дата размещения объявления      0.00
Цена                            0.00
Кол-во просмотров               0.00
Скрыто                          0.00
Объем двигателя                 0.01
Тип двигателя                   0.00
Мощность                        0.01
Коробка передач                 0.00
Привод                          0.00
Пробег                          0.80
Руль                            0.04
Поколение                       0.01
Рестайлинг                      0.01
Цвет                            0.38
Комплектация                    0.22
Владелец                        0.00
Особые отметки                 93.42
Тип кузова                      0.40
VIN                            99.11
Проверено                     100.00
Номер кузова                   99.99
Метка                           0.00
Город                           0.00
Регион                          0.00
М

Заметим, что такие данные как высота подъема, объем ковша, высота седла, тип кабины и др. на 100% отсутствуют. Это связано с тем, что в данном датасете только легковые автомобили. Сможем смело удалять эти столбцы

3. Очистка данных

In [11]:
# Перед удалением создадим отдельный столбец, тк особые отметки - очень важный признак
df['Есть особые отметки'] = df['Особые отметки'].notna().astype(int)
# Задаем порог: если пропусков больше, чем 50% - смело удаляем столбец
threshold = len(df) * 0.5  
df_cleaned = df.dropna(thresh=threshold, axis=1).copy()
print('Было колонок: ', df.shape[1])
print('Стало колонок: ', df_cleaned.shape[1])

Было колонок:  59
Стало колонок:  27


In [12]:
# Удаляем строчки, у которых отсутствует пробег - всего 0.77 от датасета, это ни на что не повлияет
df_cleaned = df_cleaned.dropna(subset=['Пробег'])

# Избавляемся от дубликатов
df_cleaned.drop_duplicates(inplace=True)

In [13]:
# Убираем ненужны столбцы
cols_to_drop = ['Дата размещения объявления', 'Кол-во просмотров', 'Скрыто', 'Ссылка', 'Владелец', 'Пропуски в данных']
df_cleaned.drop(columns=cols_to_drop, inplace=True)

In [14]:
cols_to_unknown = ['Цвет', 'Комплектация']
for col in cols_to_unknown:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')

In [15]:
# Переименовываем "Метку" в понятную "Марку"
df_cleaned.rename(columns={'Метка': 'Марка'}, inplace=True)

4. Разделение на train и test

Критический шаг. Делаем это для избежания утечки данных

In [16]:
X = df_cleaned.drop(columns=[CONFIG['TARGET']])
y = df_cleaned[CONFIG['TARGET']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['RANDOM_STATE'])

5. Контекстная очистка

In [17]:
# Заполним медианой столбцы с числовыми признаками

num_cols = ['Мощность', 'Объем двигателя']
for col in num_cols:
    global_median = X_train[col].median()
    group_medians = X_train.groupby('Название машины')[col].median()
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_medians)
        ).fillna(global_median)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_medians)
    ).fillna(global_median)

In [18]:
# Заполним модой столбцы с категориальными признаками

cat_cols = ['Привод', 'Руль', 'Тип кузова', 'Тип двигателя', 'Владельцы', 'Коробка передач']
for col in cat_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby('Название машины')[col].apply(
        lambda x: x.mode().get(0, global_mode)
    )
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_modes)
    ).fillna(global_mode)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_modes)
    ).fillna(global_mode)

In [19]:
# Особенный случай - поколение и рестайлинг зависят и от модели, и от года выпуска
special_cols = ['Поколение', 'Рестайлинг']
for col in special_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby(['Название машины', 'Год'])[col].apply(
        lambda x: x.mode().get(0, global_mode)
    ).rename(f'mode_{col}')

    X_train[col] = X_train[col].fillna(
        X_train.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

    X_test[col] = X_test[col].fillna(
        X_test.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

In [20]:
# Проверим
display(X_test.isna().sum())
X_train.isna().sum()

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

In [24]:
train_full = pd.concat([X_train, y_train], axis=1)
test_full = pd.concat([X_test, y_test], axis=1)

train_full.to_parquet('../data/processed/train_cleaned.parquet')
test_full.to_parquet('../data/processed/test_cleaned.parquet')